In [0]:
%python

from pyspark.sql.functions import *
import json

## Create & Fetch Parameters

In [0]:
%python

dbutils.widgets.text("config", "{}")

config = json.loads(dbutils.widgets.get("config"))

## Last Load Date

In [0]:
%python

from datetime import datetime

source_table = config["source_table"]
target_table = config["target_table"]
target_schema = config["target_schema"]
cdc_col = config["cdc_col"]
backdated_refresh = config.get("backdated_refresh", "").strip()

# Last Load Date logic
INITIAL_LOAD = datetime(1900, 1, 1, 0, 0, 0)

if backdated_refresh:
    last_load = backdated_refresh

else:
    target = (f"flights_project.{target_schema}.{target_table}")

    if spark.catalog.tableExists(target):
        last_load = spark.sql(f"""
            SELECT MAX({cdc_col})
            FROM {target}
        """).first()[0]             

        # Handle an existing but empty target table
        if last_load is None:
            last_load = INITIAL_LOAD

    else:
        last_load = INITIAL_LOAD

print(f"Last load timestamp: {last_load}")

## Loop through dimensions to join each dimension dynamically.

In [0]:
%python

last_load_date = last_load
fact_columns = config["fact_columns"]
dimensions = config["dimensions"]

# 1. Read the Silver source and filter incremental records
df_source = spark.table(source_table)

## Initial Load
if backdated_refresh or last_load_date == INITIAL_LOAD:
    df_source = df_source.filter(col(cdc_col) >= lit(last_load_date).cast("timestamp"))

## Normal Incremental Load
else:
    df_source = df_source.filter(col(cdc_col) >= lit(last_load_date).cast("timestamp"))

# Initialize the fact df with the filtered source
df_fact = df_source

# 2. Dynamically join each dimension
for dim in dimensions:

    fact_key = dim["fact_join_col"]
    dim_key = dim["dimension_join_col"]
    surrogate_key = dim["surrogate_key"]

    df_fact = (
        df_fact.alias("f")
        .join(
            spark.table(dim["table"])
                .select(dim_key, surrogate_key)
                .alias("d"),
            col(f"f.{fact_key}") == col(f"d.{dim_key}"),               
            "left"
        )
        .select(
            "f.*",
            col(f"d.{surrogate_key}").alias(surrogate_key)
        )
    )

# 3. Select fact table columns
surrogate_keys = [dim["surrogate_key"] for dim in dimensions]

df_fact = df_fact.select(
    *surrogate_keys,
    *fact_columns
)

## UPSERT

In [0]:
%python

from delta.tables import DeltaTable

# Primary key from config for MERGE condition
primary_key = config["primary_key"]

if spark.catalog.tableExists(f"flights_project.{target_schema}.{target_table}"):
    dlt_object = DeltaTable.forName(spark, f"flights_project.{target_schema}.{target_table}")
    (dlt_object.alias("trg")
                .merge(df_fact.alias("src"), f"src.{primary_key} = trg.{primary_key}")
                .whenMatchedUpdateAll(condition = f"src.{cdc_col} >= trg.{cdc_col}")
                .whenNotMatchedInsertAll()
                .execute())
else:
    df_fact.write\
        .format("delta")\
        .mode("append")\
        .saveAsTable(f"flights_project.{target_schema}.{target_table}")
